# Run survival pipeline locally

Same commands as `bash_scripts/launch_survival.sh` and `array_survival_run.sh`, just invoked from a notebook so you can iterate without SLURM.

**Before running:** make sure the active kernel is the `survlatent_ode` conda env (or activate it inline with `conda run -n survlatent_ode python …`). Edit the `Paths` cell to point at your data.

## Paths

Defaults are the cluster paths from `bash_scripts/build_prediction_inputs.sh`. Override `PROJECT_ROOT`, `DATA`, `V3_LABELS`, `INPUTS_DIR`, `OUTPUT_DIR` for local copies.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
DATA         = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/longitudinal_prediction_data.csv")
V3_LABELS    = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/v3_outputs/LLM_v3_labels.tsv")
INPUTS_DIR   = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis/prediction_inputs")
OUTPUT_DIR   = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis/local_runs")

SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis"

os.chdir(PROJECT_ROOT)
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BLAS thread caps (mirror the bash scripts so models don't oversubscribe cores).
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

print("cwd:        ", os.getcwd())
print("inputs_dir: ", INPUTS_DIR)
print("output_dir: ", OUTPUT_DIR)

## 1. Build prediction inputs

Required after any change to `build_prediction_inputs.py` (e.g. the `landmark_day` fix). Writes `aggregated_landmark{D}.csv`, `longitudinal_landmark{D}.csv`, etc. into `INPUTS_DIR`. Skip this cell if the inputs already exist for landmarks 0 and 90.

In [ ]:
!python {SURVIVAL_DIR}/build_prediction_inputs.py \
    --data {DATA} \
    --v3-labels-path {V3_LABELS} \
    --output-dir {INPUTS_DIR} \
    --landmark-days 0 90 \
    --time-unit-days 7 \
    --test-frac 0.20 \
    --val-frac 0.20 \
    --min-patient-coverage 0.20

## 2. Run all platinum tasks at landmarks 0 and 90

Runs cox + xgboost + deephit (PLATINUM and competing configs) at both landmarks. Skips any task whose metrics CSV already exists at the expected output path. Set `FORCE_RERUN = True` to override and rerun everything.

In [ ]:
import subprocess
import time

N_FOLDS = 5
FORCE_RERUN = False

# (model, landmark, config_dir, metrics_filename)
# config_dir is the subfolder under landmark_{D}/ where outputs are written.
TASKS = [
    ("cox",     0,  "both",      "cox_agg_multivariable_metrics.csv"),
    ("cox",     90, "both",      "cox_agg_multivariable_metrics.csv"),
    ("xgboost", 0,  "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost", 90, "both",      "landmark_xgboost_metrics.csv"),
    ("deephit", 0,  "PLATINUM",  "dynamic_deephit_metrics_PLATINUM.csv"),
    ("deephit", 90, "PLATINUM",  "dynamic_deephit_metrics_PLATINUM.csv"),
    ("deephit", 0,  "competing", "dynamic_deephit_metrics_competing.csv"),
    ("deephit", 90, "competing", "dynamic_deephit_metrics_competing.csv"),
]


def build_command(model, landmark, config_dir, row_output_dir):
    if model == "cox":
        return [
            "python", str(SURVIVAL_DIR / "cox_aggregated.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--analysis", config_dir,           # "both" runs univariate + multivariable
            "--endpoints", "platinum", "death",
            "--n-folds", str(N_FOLDS),
        ]
    if model == "xgboost":
        return [
            "python", str(SURVIVAL_DIR / "landmark_xgboost.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum", "death",
            "--n-folds", str(N_FOLDS),
        ]
    if model == "deephit":
        return [
            "python", str(SURVIVAL_DIR / "dynamic_deephit.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-day", str(landmark),
            "--config", config_dir,
            "--n-folds", str(N_FOLDS),
        ]
    raise ValueError(f"Unknown model: {model}")


summary = []
for model, landmark, config_dir, metrics_filename in TASKS:
    row_output_dir = OUTPUT_DIR / model / f"landmark_{landmark}" / config_dir
    metrics_path = row_output_dir / metrics_filename
    tag = f"{model:8s}  landmark_{landmark:<2}  {config_dir}"

    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[skip] {tag}  ->  {metrics_path.relative_to(OUTPUT_DIR)} exists")
        summary.append((tag, "skipped", 0.0))
        continue

    row_output_dir.mkdir(parents=True, exist_ok=True)
    cmd = build_command(model, landmark, config_dir, row_output_dir)
    print(f"[run ] {tag}")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    elapsed = time.time() - t0
    status = "ok" if rc == 0 else f"FAILED (rc={rc})"
    print(f"[done] {tag}  -> {status}  ({elapsed/60:.1f} min)\n")
    summary.append((tag, status, elapsed))

print("\n=== run summary ===")
for tag, status, elapsed in summary:
    print(f"  {tag}  {status:>20s}  {elapsed/60:6.1f} min")

## 3. Inspect outputs

Pulls the headline metrics row from each task's metrics CSV into one comparison table.

In [ ]:
import pandas as pd

rows = []
for model, landmark, config_dir, metrics_filename in TASKS:
    metrics_path = OUTPUT_DIR / model / f"landmark_{landmark}" / config_dir / metrics_filename
    if not metrics_path.exists():
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "-", "n_test": None, "n_test_events": None,
            "c_index": None, "mean_auc_t": None, "integrated_brier": None,
            "status": "missing",
        })
        continue
    df = pd.read_csv(metrics_path)

    if model == "cox":
        platinum = df[df["endpoint"] == "platinum"].iloc[0]
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "platinum",
            "n_test": int(platinum["n_test"]),
            "n_test_events": int(platinum["n_events_test"]),
            "c_index": float(platinum["test_c_index"]),
            "mean_auc_t": float(platinum["test_mean_auc_t"]),
            "integrated_brier": float(platinum["test_integrated_brier"]),
            "status": "ok",
        })
    elif model == "xgboost":
        platinum = df[df["endpoint"] == "platinum"].iloc[0]
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "platinum",
            "n_test": int(platinum["n_test"]),
            "n_test_events": int(platinum["n_test_events"]),
            "c_index": float(platinum["c_index"]),
            "mean_auc_t": float(platinum["mean_auc_t"]),
            "integrated_brier": float(platinum["integrated_brier"]),
            "status": "ok",
        })
    elif model == "deephit":
        for _, r in df.iterrows():
            rows.append({
                "model": model, "landmark": landmark, "config": config_dir,
                "endpoint": str(r["event"]),
                "n_test": int(r["n_test"]),
                "n_test_events": int(r["n_test_events"]),
                "c_index": float(r["c_index"]),
                "mean_auc_t": float(r["mean_auc_t"]),
                "integrated_brier": float(r["integrated_brier"]),
                "status": "ok",
            })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["endpoint", "landmark", "model", "config"]).reset_index(drop=True)
summary_df